# Train a mapper on the CV 27.0 speaker split (Colab GPU)

**Settings**
- `LAYER`: the wav2vec2 layer, as an index into `outputs.hidden_states`. 0 is the feature
  projection, k is transformer block k, and 24 includes the final LayerNorm.
- `CONFIG`: **`tempo`** (option A, tempo-warped targets) is the standard for all new runs.
  `linear` is the old linear stretch, kept only for the existing layer-9 comparison run.

**Input normalisation** (`configs/train.yaml` `data.input_norm: true`; `docs/ablation_plan.md`,
Amendment 1): the mapper input is normalised per dimension with mean/std computed per layer on
the **train split only**. The statistics are stored in every checkpoint and in `input_norm.npz`,
and applied inside the model in training and inference.

Everything else comes from `configs/train.yaml` and is identical across runs:
- batch size 4
- Adam, lr 1e-3, ReduceLROnPlateau (patience 5)
- 50 epochs, seed 42
- severity `severe`

The fixed-layer ablation is pre-registered in `docs/ablation_plan.md`. It covers layers
3, 6, 9, 12, 18 and 24; layer 9 is retrained too, so all six use identical code.

**Run folder.** `runs/cv27_layer<NN>_<CONFIG>` with a two-digit layer, for example
`runs/cv27_layer03_tempo`, or `runs/cv27_layer09_tempo` for the retrained layer 9.
The W2 runs `runs/cv27_layer9_tempo` and `runs/cv27_layer9_linear` were trained without
input normalisation. This notebook never writes to them.

**Data in `DRIVE_DATA`**
- From the Mac (`~/data/cv-th/colab/`): `cv27_mel_cache.tar`, `cv27_metadata.tar.gz` and
  `SHA256SUMS`.
- The embeddings for `LAYER`:
  - Layer 9: `cv27_embeddings_layer09.tar`, uploaded from the Mac.
  - Other layers: `cv27_embeddings_layer<NN>.tar` plus its `.sha256`, written to Drive by
    `notebooks/colab_extract_layers.ipynb`.

The data is copied from Drive to `/content` and trained from there. After every epoch a
full resumable checkpoint (`latest.pt`) is written to Drive. It holds the model, optimizer,
scheduler, epoch, best val loss and RNG state. If the session restarts, run all cells again
and training resumes from `latest.pt` automatically.

In [ ]:
# ---- settings -------------------------------------------------------------
LAYER = 3                 # wav2vec2 hidden_states index: 3, 6, 9, 12, 18 or 24 for the ablation
CONFIG = "tempo"          # "tempo" = option A (standard); "linear" = old linear stretch (layer-9 comparison only)

REPO_URL = "https://github.com/KankaveeRamsri/thai-dsr.git"
REF = "w3-layer-ablation" # branch, tag or commit sha to train (the layer-9 runs used w1-speaker-split / ef6b427)

DRIVE_ROOT = "/content/drive/MyDrive/thai-dsr"
DRIVE_DATA = f"{DRIVE_ROOT}/cv27_data"                      # the uploaded archives
RUN_NAME = f"cv27_layer{LAYER:02d}_{CONFIG}"   # layer 9 -> cv27_layer09_tempo (W2 keeps cv27_layer9_tempo)
DRIVE_RUN = f"{DRIVE_ROOT}/runs/{RUN_NAME}"                 # checkpoints and logs

DATA_ROOT = "/content/cv27"   # local copy of the data (fast disk, not Drive)
REPO_DIR = "/content/thai-dsr"
ALLOW_COMMIT_CHANGE = False   # refuse to resume a run with code from a different commit

assert CONFIG in ("linear", "tempo"), CONFIG
assert isinstance(LAYER, int) and 0 <= LAYER <= 24, LAYER
print("run:", DRIVE_RUN)

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| torch", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REF], check=True)
branch = subprocess.run(["git", "-C", REPO_DIR, "symbolic-ref", "-q", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
if branch:  # a branch: bring it up to date; a commit sha stays detached as given
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--quiet", "--hard", f"origin/{branch}"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
os.chdir(REPO_DIR)
print("repo", REPO_DIR, "| ref", REF, "| commit", COMMIT)

In [ ]:
# torch is preinstalled on Colab and satisfies requirements.txt, so it is not reinstalled.
!pip install -q -r requirements.txt

In [ ]:
import hashlib
import shutil
import time

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()

EMB_ARCHIVE = f"cv27_embeddings_layer{LAYER:02d}.tar"
ARCHIVES = [EMB_ARCHIVE, "cv27_mel_cache.tar", "cv27_metadata.tar.gz"]
expected = {name: digest for digest, name in
            (line.split() for line in open(f"{DRIVE_DATA}/SHA256SUMS") if line.strip())}
if os.path.exists(f"{DRIVE_DATA}/{EMB_ARCHIVE}.sha256"):  # written by colab_extract_layers.ipynb
    expected[EMB_ARCHIVE] = open(f"{DRIVE_DATA}/{EMB_ARCHIVE}.sha256").read().split()[0]
assert EMB_ARCHIVE in expected, f"no checksum for {EMB_ARCHIVE}: run colab_extract_layers.ipynb first"

# Mel targets and metadata are shared by every layer; the embeddings are per layer.
marker = f"{DATA_ROOT}/.extracted"
emb_marker = f"{DATA_ROOT}/.extracted_layer{LAYER:02d}"
if not (os.path.exists(marker) and os.path.exists(emb_marker)):
    local = "/content/archives"
    os.makedirs(local, exist_ok=True)
    os.makedirs(f"{DATA_ROOT}/meta", exist_ok=True)
    todo = [EMB_ARCHIVE] if os.path.exists(marker) else ARCHIVES
    for name in todo:
        start = time.time()
        shutil.copy(f"{DRIVE_DATA}/{name}", f"{local}/{name}")
        assert sha256(f"{local}/{name}") == expected[name], f"checksum mismatch: {name}"
        print(f"copied + verified {name} in {time.time() - start:.0f}s")
    subprocess.run(["tar", "-xf", f"{local}/{EMB_ARCHIVE}", "-C", DATA_ROOT], check=True)
    open(emb_marker, "w").close()
    if not os.path.exists(marker):
        subprocess.run(["tar", "-xf", f"{local}/cv27_mel_cache.tar", "-C", DATA_ROOT], check=True)
        subprocess.run(["tar", "-xzf", f"{local}/cv27_metadata.tar.gz", "-C", f"{DATA_ROOT}/meta"], check=True)
        open(marker, "w").close()
    shutil.rmtree(local)

counts = {"embeddings": len([f for f in os.listdir(f"{DATA_ROOT}/embeddings") if f.endswith(f"_layer{LAYER:02d}.npy")]),
          **{d: len(os.listdir(f"{DATA_ROOT}/{d}")) for d in ("mel_cache/clean", "mel_cache/tempo")}}
print(counts)
assert counts == {"embeddings": 3612, "mel_cache/clean": 2996, "mel_cache/tempo": 3612}, counts

# The archived metadata must match the code's tracked copies (same split, same tempo segments).
for name in ["cv27_th_speaker_split.csv", "cv27_th_splits.json",
             "cv27_th_distortion_params.csv", "cv27_th_clean_16k_index.csv"]:
    assert sha256(f"{DATA_ROOT}/meta/{name}") == sha256(f"data/manifests/{name}"), name
print("metadata matches data/manifests/ at", COMMIT[:7])

In [ ]:
import json

os.environ["DATA_ROOT"] = DATA_ROOT  # paths in data/manifests/*.csv are relative to it
subprocess.run(["python", "scripts/build_cv_training_manifest.py",
                "--manifest", f"{DATA_ROOT}/training_manifest.csv",
                "--splits", f"{DATA_ROOT}/splits.json"], check=True)
assert json.load(open(f"{DATA_ROOT}/splits.json"))["splits"] == \
       json.load(open("data/manifests/cv27_th_splits.json"))["splits"]

In [ ]:
import yaml

train_cfg = yaml.safe_load(open("configs/train.yaml"))
FIXED = {
    "layer": train_cfg["data"]["layer"], "severity": train_cfg["data"]["severity"],
    "batch_size": train_cfg["optim"]["batch_size"], "learning_rate": train_cfg["optim"]["learning_rate"],
    "optimizer": train_cfg["optim"]["optimizer"], "weight_decay": train_cfg["optim"]["weight_decay"],
    "lr_scheduler": train_cfg["optim"]["lr_scheduler"], "num_epochs": train_cfg["optim"]["num_epochs"],
    "seed": train_cfg["runtime"]["seed"], "input_norm": train_cfg["data"].get("input_norm"),
}
assert FIXED["layer"] == 9, "configs/train.yaml changed: the layer is set with LAYER, not the yaml"
FIXED["layer"] = LAYER  # passed to train.py as --layer
print(json.dumps(FIXED, indent=2))
assert FIXED == {"layer": LAYER, "severity": "severe", "batch_size": 4, "learning_rate": 0.001,
                 "optimizer": "adam", "weight_decay": 0.0, "lr_scheduler": "reduce_on_plateau",
                 "num_epochs": 50, "seed": 42, "input_norm": True}, "configs/train.yaml changed: every run must share it"

os.makedirs(DRIVE_RUN, exist_ok=True)
info_path = f"{DRIVE_RUN}/run_info.json"
info = json.load(open(info_path)) if os.path.exists(info_path) else {
    "config": CONFIG, "layer": LAYER, "input_norm": True, "commit": COMMIT, "fixed": FIXED, "sessions": []}
if info.get("layer", 9) != LAYER or info["config"] != CONFIG or info.get("input_norm") is not True:
    raise RuntimeError(f"{DRIVE_RUN} belongs to layer {info.get('layer', 9)}, config {info['config']}, "
                       f"input_norm {info.get('input_norm')}: not this run (W2 folders are never reused)")
if info["commit"] != COMMIT and os.path.exists(f"{DRIVE_RUN}/latest.pt") and not ALLOW_COMMIT_CHANGE:
    raise RuntimeError(f"{DRIVE_RUN} was started at commit {info['commit']}, now {COMMIT}. "
                       "Set REF to that commit, or ALLOW_COMMIT_CHANGE = True.")
info["sessions"].append({"started": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": GPU_NAME, "commit": COMMIT})
json.dump(info, open(info_path, "w"), indent=2)

In [ ]:
import sys

cmd = [sys.executable, "-u", "-m", "src.training.train",
       "--manifest", f"{DATA_ROOT}/training_manifest.csv",
       "--splits", "data/manifests/cv27_th_splits.json", "--fixed-splits",
       "--embeddings-dir", f"{DATA_ROOT}/embeddings",
       "--mel-cache-dir", f"{DATA_ROOT}/mel_cache", "--preload", "--num-workers", "0",
       "--alignment", CONFIG, "--layer", str(LAYER), "--device", "cuda",
       "--checkpoint-dir", f"{DRIVE_RUN}/checkpoints", "--log-dir", f"{DRIVE_RUN}/logs",
       "--best-checkpoint", f"{DRIVE_RUN}/best.pt",
       "--loss-curve", f"{DRIVE_RUN}/logs/loss_curve.png",
       "--save-latest", f"{DRIVE_RUN}/latest.pt", "--auto-resume"]
if CONFIG == "tempo":
    cmd += ["--distortion-params", "data/manifests/cv27_th_distortion_params.csv"]

print("GPU:", GPU_NAME, "| layer:", LAYER, "| config:", CONFIG, "| commit:", COMMIT[:7])
env = dict(os.environ, TQDM_DISABLE="1")  # one line per epoch instead of progress bars
with open(f"{DRIVE_RUN}/train.log", "a") as log:
    log.write(f"\n===== session {time.strftime('%Y-%m-%d %H:%M:%S')} on {GPU_NAME} at {COMMIT}\n")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    assert process.wait() == 0, f"train.py exited with {process.returncode}"

In [ ]:
import re

latest = torch.load(f"{DRIVE_RUN}/latest.pt", map_location="cpu", weights_only=True)
print(f"layer {LAYER}, config {CONFIG}: epoch {latest['epoch']}/{FIXED['num_epochs']}, "
      f"best val loss {latest['best_val_loss']:.4f} at epoch {latest.get('best_epoch')}")
times = [float(m.group(1)) for m in re.finditer(r"^epoch \d+ time ([0-9.]+)s",
                                                open(f"{DRIVE_RUN}/train.log").read(), re.M)]
norm = latest.get("input_norm") or {}
print(f"input normalisation: layer {norm.get('layer')}, {norm.get('split')} split, "
      f"{norm.get('n_clips')} clips, {norm.get('n_frames')} frames, "
      f"{'saved' if os.path.exists(f'{DRIVE_RUN}/input_norm.npz') else 'MISSING'} input_norm.npz")
assert norm.get("split") == "train" and norm.get("layer") == LAYER and norm.get("n_clips") == 2383, norm
if times:
    print(f"time per epoch on {GPU_NAME}: mean {sum(times) / len(times):.1f}s over {len(times)} epoch(s)")